# Report - Computer Vision and Deep Learning

`Francisco Bley Ruthes`


# Import the libs you need


In [ ]:
# Standard library
import os
import re
import gc
import math
import time
import hashlib
from pathlib import Path

# Images, arrays, tables and plots
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Classical features and machine learning models
from sklearn.model_selection import train_test_split
from skimage.feature import hog, local_binary_pattern
import joblib
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             ConfusionMatrixDisplay, classification_report)
from sklearn.utils.class_weight import compute_class_weight

# Data augmentation
import albumentations as A

# Deep learning (the backend must be set before importing keras)
os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch


# 1. Define the task and dataset


Why this problem?

Before starting this dataset analysis, I need to explain why I chose it. Since agriculture is the main activity for the survival of human society, I started researching datasets related to plant diseases. However, most of them were related to only one specific plant, and the differences between the images were sometimes microscopic. For the purpose of this project, I needed to find images of classes that were very different from each other in most cases, and I wanted to cover more plants and have more impact. Therefore, I chose to work with insects, because they affect not just one type of crop.

Task and dataset

Dataset: Agricultural Pests Image Dataset (Kaggle) - https://www.kaggle.com/datasets/vencerlanz09/agricultural-pests-image-dataset

Task: multiclass image classification. Given an image of an agricultural pest, predict its class among 12 classes: ants, bees, beetle, catterpillar, earthworms, earwig, grasshopper, moth, slug, snail, wasp and weevil.

The dataset has 5,494 images organized in one folder per class, with no pre-defined split. A local copy was archived for this project.

# 2. Acquire the dataset


In [ ]:
# Functions to help us prit images correctly

def imread_unicode(path):
    """cv2.imread fails on Windows paths with accents.
    Workaround: read the bytes manually and decode them."""
    data = np.fromfile(path, dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

def show_image(img_bgr, title=None, ax=None):
    """Show a cv2 image (BGR) with matplotlib."""
    target = ax if ax is not None else plt
    target.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    if ax is not None:
        ax.axis("off")
        if title:
            ax.set_title(title, fontsize=9)
    else:
        plt.axis("off")
        if title:
            plt.title(title)

We will start by creating our consulting dataset, which will give us all the information we need to start exploring the nature of our data.

In [ ]:
# Directory main path, the dataset folder is next to the notebook
# The environment variable PEST_DATA_DIR can point to another folder
DATA_DIR = Path(os.environ.get("PEST_DATA_DIR", "Agricultural Pests Image Dataset"))
assert DATA_DIR.exists(), f"Dataset folder not found, put the folder '{DATA_DIR}' next to the notebook"

# Classes are directories in the 'Agricultural Pests Image Dataset'
CLASSES = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"{len(CLASSES)} classes:", CLASSES)


In [ ]:
# Defining valid extensions
VALID_SUFIX = {".jpg", ".jpeg", ".png"}

# Address dictionaries
records = [
    {"file_path": str(f), "label": cls}
    for cls in CLASSES
    for f in (DATA_DIR / cls).iterdir()
    if f.suffix.lower() in VALID_SUFIX
]

# Our dataset
df = pd.DataFrame(records)
print(df.shape)
df.head()

In [ ]:
# Metadata extraction
def read_info(path):
    img = imread_unicode(path)
    if img is None:
        return pd.Series({"height": np.nan, "width": np.nan, "channels": np.nan})
    h,w,c = img.shape
    return pd.Series({"height": h, "width": w, "channels": c})

df["file_name"] = df["file_path"].apply(lambda p: Path(p).stem)
df[["height", "width", "channels"]] = df["file_path"].apply(read_info)
df["file_type"] = df["file_path"].apply(lambda p: Path(p).suffix.lower())
df["file_size_kb"] = df["file_path"].apply(lambda p: Path(p).stat().st_size / 1024)

# Checking for corrupted files (nan or null values)
corrupted_files = df[["height", "width", "channels"]].isna().sum(axis=1)
print(f"{corrupted_files} corrupted files.")

df = df.dropna(subset=["height", "width", "channels"]).reset_index(drop=True)
df["height"] = df["height"].astype(int)
df["width"] = df["width"].astype(int)
df["channels"] = df["channels"].astype(int)

df.head()

In [ ]:
# Testing image print
fig, axes = plt.subplots(1,2,figsize=(8,4))

img_test = df.sample(1, random_state=14).iloc[0] # Returns a line, not the dataframe
img_test2 = df.sample(1, random_state=17).iloc[0] 

show_image(imread_unicode(img_test["file_path"]), title=f"{img_test.label} image", ax=axes[0])
show_image(imread_unicode(img_test2["file_path"]), title=f"{img_test2.label} image", ax=axes[1])

plt.show()

# 3. Dataset analysis and visualization


In [ ]:
# Basic Statistics
imgs_count = len(df)
print(f"\nTotal images: {imgs_count}")

# File type counting
print("\nTypes counting:")
print(df["file_type"].value_counts())

# Images per class
print("\nImages per class:")
print(df["label"].value_counts())


In [ ]:
# Images per class
class_counts = df["label"].value_counts()
class_pct = (class_counts / len(df) * 100).round(1)

class_table = pd.DataFrame({"count": class_counts, "percent": class_pct})
print(class_table)
print(f"\nTotal: {class_counts.sum()} | Mean per class: {class_counts.mean():.1f}")
print(f"Max/min ratio: {class_counts.max() / class_counts.min():.2f}x")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(class_counts.index, class_counts.values, color="#4C72B0", edgecolor="white")
ax.bar_label(bars)
ax.axhline(class_counts.mean(), color="red", linestyle="--", label="mean")
ax.set_title("Images per class")
ax.set_ylabel("Number of images")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

Notice that we have 135 fewer images for earthworms, 67 fewer for slug, 42 fewer for beetle and 24 fewer for caterpillar, all compared to the mean, which is 458 images per class. The other classes are above the mean, by 8 to 42 images. This could make the models learn more about the classes above the mean and less about the classes below it, because of the variability.

In [ ]:
# Funtion to show a grid of images from a dataframe
def show_grid(sub_df, ncols=6, title_fn=None, suptitle=None):
    n = len(sub_df)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    axes = np.array(axes).flatten()
    for ax, (_, row) in zip(axes, sub_df.iterrows()):
        title = title_fn(row) if title_fn else row["label"]
        show_image(imread_unicode(row["file_path"]), title=title, ax=ax)
    for ax in axes[n:]:
        ax.axis("off")
    if suptitle:
        plt.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

# Representative samples
samples = df.groupby("label").sample(5, random_state=42)
for cls in CLASSES:
    show_grid(samples[samples["label"] == cls], ncols=5, suptitle=cls) # 5 random images per class

These samples show the high variability in this dataset. We see the same insects in different positions and perspectives, and on surfaces with different colors. This can help our trained models to generalize better. However, I need to emphasize that the images look very clear and clean. In real conditions, the background is usually more cluttered, and detection may be harder depending on the location or the crop. For some pests, using a standardized trap can make detection easier, since it gives a uniform background and more controlled conditions.

In [ ]:
# Duplicate detection (same file hash)
df["hash"] = df["file_path"].apply(lambda p: hashlib.md5(open(p,"rb").read()).hexdigest())

duplicates = df["hash"].duplicated(keep=False).sum()

print(f"\nDuplicates: {duplicates}")

dup_df = df[df["hash"].duplicated(keep=False)].sort_values("hash")
print(dup_df[["file_name"]])

for h, group in dup_df.groupby("hash"):
    print(f"\nHash: {h} ({len(group)} files)")
    print(group[["file_name"]])

print("\n")

# Duplicates across classes
cross_class = dup_df.groupby("hash")["label"].nunique()
n_cross = (cross_class > 1).sum()
print(f"Duplicate groups: {cross_class.shape[0]}")
print(f"Groups across different classes: {n_cross}")

if n_cross > 0:
    print("\nProblematic groups (same image in different classes):")
    print(dup_df[dup_df["hash"].isin(cross_class[cross_class > 1].index)][["file_name", "label", "hash"]])
else:
    print("No duplicates across classes.")

We didnt get to many duplicated cases, this would not be a problem. The only thing we can do about it is to see the duplciated images side by side just to confirm, but we now that since the hash is the same because of the bytes combination, it is know that tehy are the same and we can just delete one of them for training or just maintain it on the same group of the training split.

In [ ]:
# Visualizing duplicated images
n = len(dup_df)
ncols = 6                      # images per row
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
axes = axes.flatten()          # 2D grid -> 1D list

for ax, (_, row) in zip(axes, dup_df.iterrows()):
    show_image(imread_unicode(row["file_path"]),
               title=f'{row["label"]}\n{row["width"]}x{row["height"]}\nExt: {row["file_type"]}', ax=ax)

for ax in axes[n:]:            # hide unused axes
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Dimension and file size statistics
df["aspect_ratio"] = df["width"] / df["height"]

print("Overall describe (numeric):")
print(df[["height", "width", "aspect_ratio", "file_size_kb"]].describe())

print("\nClass imbalance (max/min count):")
counts = df["label"].value_counts()
print(f"Max: {counts.idxmax()} ({counts.max()}) | Min: {counts.idxmin()} ({counts.min()}) | Ratio: {counts.max() / counts.min():.2f}x")

print("\nStats per class (mean/std):")
per_class = df.groupby("label")[["height", "width", "aspect_ratio", "file_size_kb"]].agg(["mean", "std"])
print(per_class)

print("\nExtreme resolutions:")
print(f"Min height: {df.loc[df['height'].idxmin(), ['file_name', 'label', 'height', 'width']].to_dict()}")
print(f"Max height: {df.loc[df['height'].idxmax(), ['file_name', 'label', 'height', 'width']].to_dict()}")
print(f"Min width: {df.loc[df['width'].idxmin(), ['file_name', 'label', 'height', 'width']].to_dict()}")
print(f"Max width: {df.loc[df['width'].idxmax(), ['file_name', 'label', 'height', 'width']].to_dict()}")

print("\nLargest/smallest file on disk:")
print(f"Smallest: {df.loc[df['file_size_kb'].idxmin(), ['file_name', 'label', 'file_size_kb']].to_dict()}")
print(f"Largest: {df.loc[df['file_size_kb'].idxmax(), ['file_name', 'label', 'file_size_kb']].to_dict()}")

print("\nChannel count (expected: all 3, RGB):")
print(df["channels"].value_counts())

In [ ]:
# Dimension and resolution histograms

# Helper: write the median value on top of each box
def annotate_medians(ax, data, fmt="{:.0f}"):
    for i, d in enumerate(data, start=1):
        m = d.median()
        ax.text(i, m, fmt.format(m), ha="center", va="bottom", fontsize=7, color="red")

# Overall distribution of the whole dataset (dashed line = mean)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ["height", "width", "aspect_ratio", "file_size_kb"]):
    mean = df[col].mean()
    ax.hist(df[col], bins=30, color="#4C72B0", edgecolor="white")
    ax.axvline(mean, color="black", linestyle="--", label=f"mean={mean:.2f}")
    ax.set_title(col)
    ax.legend()
plt.suptitle("Overall dataset distribution")
plt.tight_layout()
plt.show()

# Class comparison (red number = median of each class)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ["height", "width", "aspect_ratio"]):
    data = [df.loc[df["label"] == c, col] for c in CLASSES]
    ax.boxplot(data, tick_labels=CLASSES)
    annotate_medians(ax, data, fmt="{:.2f}" if col == "aspect_ratio" else "{:.0f}")
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=90)
plt.suptitle("Class comparison")
plt.tight_layout()
plt.show()

Related to image size, the mean height and width are consistent across all classes. The aspect ratio (width/height) is always above 1, around 1.3, which is expected since most of the images were taken in landscape orientation.

The width has almost no variation: most images have a width of 300 px, and the few different values appear as outliers. The height varies more, mainly in catterpillar and moth, which show a higher standard deviation and fewer outliers in general compared to width.

For training, it is simple to adjust the images to a square size such as 224 x 224, which makes training faster. To avoid distorting the images, we can resize each image by its larger dimension and fill the remaining space with padding of a single color (letterbox).

In [ ]:
# Brightness via perceptual luminance
def luminance_stats(path):
    img = imread_unicode(path)
    if img is None:
        return pd.Series({"luminance_mean": np.nan, "luminance_std": np.nan})
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # 0.114*B + 0.587*G + 0.299*R
    return pd.Series({"luminance_mean": gray.mean(), "luminance_std": gray.std()})

df[["luminance_mean", "luminance_std"]] = df["file_path"].apply(luminance_stats)

print("Overall describe (luminance):")
print(df[["luminance_mean", "luminance_std"]].describe())

print("\nMean luminance per class:")
print(df.groupby("label")[["luminance_mean", "luminance_std"]].mean())

print("\nDarkest/brightest images by luminance (top 5):")
print("Darkest:")
print(df.nsmallest(5, "luminance_mean")[["file_name", "luminance_mean"]])
print("\nBrightest:")
print(df.nlargest(5, "luminance_mean")[["file_name", "luminance_mean"]])

In [ ]:
# Luminance histograms

# Overall distribution of the whole dataset (dashed line = mean)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ["luminance_mean", "luminance_std"]):
    mean = df[col].mean()
    ax.hist(df[col], bins=30, color="#DD8452", edgecolor="white")
    ax.axvline(mean, color="black", linestyle="--", label=f"mean={mean:.1f}")
    ax.set_title(col)
    ax.legend()
plt.suptitle("Overall dataset distribution")
plt.tight_layout()
plt.show()

# Class comparison (red number = median of each class)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col in zip(axes, ["luminance_mean", "luminance_std"]):
    data = [df.loc[df["label"] == c, col] for c in CLASSES]
    ax.boxplot(data, tick_labels=CLASSES)
    annotate_medians(ax, data)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=90)
plt.suptitle("Class comparison")
plt.tight_layout()
plt.show()

In [ ]:
# Luminance histograms per class
overall_mean = df["luminance_mean"].mean()

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True, sharey=True)
for ax, c in zip(axes.flatten(), CLASSES):
    class_mean = df.loc[df["label"] == c, "luminance_mean"].mean()
    ax.hist(df.loc[df["label"] == c, "luminance_mean"], bins=25, color="#DD8452", edgecolor="white")
    ax.axvline(overall_mean, color="black", linestyle="--", linewidth=1)
    ax.axvline(class_mean, color="red", linewidth=1.5)
    ax.set_title(f"{c} (mean={class_mean:.0f})")
plt.suptitle(f"Luminance mean per class (black dashed = overall mean {overall_mean:.0f}, red = class mean)")
plt.tight_layout()
plt.show()

Regarding luminance, the mean shows only a small to moderate difference across classes. This difference is small compared to the variation inside each class, but it is still visible in the boxplots. The luminance_std (pixel variation inside one image) is very similar for all classes. I suspect this is because most images were taken during the day, in the crops.

Ants and weevil are the brightest classes, while slug, grasshopper and earthworms are the darkest.

This difference between classes can be a risk for the models. They could learn to associate the brightness of the image with the class, instead of the shape and texture of the insect. To reduce this risk, I plan to use brightness and contrast jitter in the data augmentation.

In [ ]:
# Sharpness (blur) via variance of the Laplacian (lower = blurrier)
def sharpness(path):
    img = imread_unicode(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

df["sharpness"] = df["file_path"].apply(sharpness)

print("Overall describe (sharpness):")
print(df["sharpness"].describe())

print("\nSharpness per class (mean/median/std):")
print(df.groupby("label")["sharpness"].agg(["mean", "median", "std"]))

In [ ]:
# Sharpness histograms and boxplot per class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mean = df["sharpness"].mean()
median = df["sharpness"].median()
axes[0].hist(df["sharpness"], bins=40, color="#55A868", edgecolor="white")
axes[0].axvline(mean, color="black", linestyle="--", label=f"mean={mean:.0f}")
axes[0].axvline(median, color="red", linestyle="-", label=f"median={median:.0f}")
axes[0].set_title("sharpness (overall)")
axes[0].legend()

data = [df.loc[df["label"] == c, "sharpness"] for c in CLASSES]
axes[1].boxplot(data, tick_labels=CLASSES)
annotate_medians(axes[1], data)
axes[1].set_title("sharpness (per class, red = median)")
axes[1].tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

Analysing the blur data by class, earthworms images are sharper than the other classes, and this is visible both in the mean and in the median. Half of the pictures have a blur value below 622, which is the median. The median is lower than the mean, which means that the distribution is right-skewed. In this case, the median represents the typical image better than the mean, because a few images with very high values pull the mean up.

In [ ]:
# Image quality: blurriest images and lowest resolution
show_grid(df.nsmallest(6, "sharpness"), ncols=6,
          title_fn=lambda r: f'{r["label"]}\nsharp={r["sharpness"]:.1f}',
          suptitle="Blurriest images")

df["n_pixels"] = df["width"] * df["height"]
show_grid(df.nsmallest(6, "n_pixels"), ncols=6,
          title_fn=lambda r: f'{r["label"]}\n{r["width"]}x{r["height"]}',
          suptitle="Lowest resolution images")

Looking at the blurriest samples, most of these pictures can still be used for training. The earwig image is also usable, but it is harder to see the insect in it. The model will need to identify the pests even when the image is not perfect. If we only give good examples, it will depend on high quality images, and real conditions will not always provide them.

About the lowest resolution images, I do not see a big problem. These images show the insect from a closer view, which is a good opportunity for the model to learn its details. The quality of the samples shown is also good. We can use them for training by applying our resize strategy with padding.

In [ ]:
# Image quality: darkest and brightest images
show_grid(df.nsmallest(6, "luminance_mean"), ncols=6,
          title_fn=lambda r: f'{r["label"]}\nlum={r["luminance_mean"]:.1f}',
          suptitle="Darkest images")
show_grid(df.nlargest(6, "luminance_mean"), ncols=6,
          title_fn=lambda r: f'{r["label"]}\nlum={r["luminance_mean"]:.1f}',
          suptitle="Brightest images")

Looking at the darkest and brightest images, it is possible to see the insects in both cases. In the bright images it is easier to see the insect. In the dark images, the insect can be confused with the background, mainly when it has a dark shell.

In [ ]:
# Intra-class variability: 8 random images from each class
for cls in CLASSES:
    show_grid(df[df["label"] == cls].sample(8, random_state=7), ncols=8, suptitle=f"Intra-class variability: {cls}")

After seeing these intra-class samples, we have some concerns. The same class can contain different species of the same insect, which makes the class more variable. There are also larger elements in some images that can steal the model's attention. For example, in one caterpillar image a bird is eating the caterpillar, and it is hard to see the caterpillar, only the bird.

In [ ]:
# Mean color histogram per class (32 bins per channel, normalized)
def channel_hist(path, bins=32):
    img = imread_unicode(path)
    n_pixels = img.shape[0] * img.shape[1]
    return np.concatenate([
        cv2.calcHist([img], [c], None, [bins], [0, 256]).flatten() / n_pixels
        for c in range(3)
    ])

hists = np.array([channel_hist(p) for p in df["file_path"]])
class_hist = {c: hists[(df["label"] == c).values].mean(axis=0) for c in CLASSES}

fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharey=True)
for ax, c in zip(axes.flatten(), CLASSES):
    h = class_hist[c]
    ax.plot(h[:32], color="blue", label="B")
    ax.plot(h[32:64], color="green", label="G")
    ax.plot(h[64:], color="red", label="R")
    ax.set_title(c)
axes[0, 0].legend()
plt.suptitle("Mean color histogram per class")
plt.tight_layout()
plt.show()

Looking at the mean color histograms, almost all classes show the same pattern: a peak toward the right side, with the green curve above the others, the red in the middle and the blue below. This suggests that the images are dominated by the background, which is similar in all classes (plants and soil), and not by the insect. Because of this, the color histogram does not separate the classes well. Color alone should be a weak feature for the classifiers, and shape and texture features should be more useful.

In [ ]:
# Inter-class similarity: distance between mean color histograms (Heat map)
names = list(class_hist.keys())
mean_hists = np.array([class_hist[c] for c in names])

dist = np.linalg.norm(mean_hists[:, None, :] - mean_hists[None, :, :], axis=2)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(dist, cmap="viridis")
ax.set_xticks(range(len(names)))
ax.set_yticks(range(len(names)))
ax.set_xticklabels(names, rotation=90)
ax.set_yticklabels(names)
plt.colorbar(im, label="Euclidean distance")
ax.set_title("Distance between class mean color histograms")
plt.tight_layout()
plt.show()

# Class pairs sorted by distance (smaller distance = more similar colors)
pair_dist = [(names[i], names[j], dist[i, j])
             for i in range(len(names)) for j in range(i + 1, len(names))]
pair_dist = sorted(pair_dist, key=lambda x: x[2])

print("Most similar pairs:")
for a, b, d in pair_dist[:5]:
    print(f"{a} - {b}: {d:.4f}")

print("\nMost different pairs:")
for a, b, d in pair_dist[-5:][::-1]:
    print(f"{a} - {b}: {d:.4f}")

The heat map shows that some classes have very similar mean color histograms, such as slug and snail, and beetle with catterpillar, earwig and wasp. On the other hand, ants stands out from several classes, mainly bees and grasshopper, which agrees with the higher brightness of ants seen in the luminance analysis. Since many pairs are close in color, color alone will not separate these classes well, and the models will need shape and texture information.

In [ ]:
# Inter-class similarity: the most similar class pairs by color (4 images each)
top_n = 10
pairs = [(a, b) for a, b, d in pair_dist[:top_n]]

for a, b in pairs:
    pair_df = pd.concat([df[df["label"] == a].sample(4, random_state=1),
                         df[df["label"] == b].sample(4, random_state=1)])
    show_grid(pair_df, ncols=4, suptitle=f"{a} (top) vs {b} (bottom)")

Looking at the 10 pairs with the most similar mean color, the insects in each pair are visually different, with different shapes and textures. So similar color does not mean similar appearance. Shape and texture are what separate these classes.

In [ ]:
# Ambiguity: extreme aspect ratios
show_grid(df.nsmallest(6, "aspect_ratio"), ncols=6,
          title_fn=lambda r: f'{r["label"]}\nAR={r["aspect_ratio"]:.2f}',
          suptitle="Lowest aspect ratios (tall images)")
show_grid(df.nlargest(6, "aspect_ratio"), ncols=6,
          title_fn=lambda r: f'{r["label"]}\nAR={r["aspect_ratio"]:.2f}',
          suptitle="Highest aspect ratios (wide images)")

The most extreme aspect ratios range from 0.31 (very tall images) to 3.41 (very wide images). The insects are still visible in these images, but stretching them to 224 x 224 would distort them, which supports the use of letterbox resizing: resizing by the larger dimension and filling the remaining space with padding of a single color.

In [ ]:
# Ambiguity: lowest contrast images (low luminance std)
show_grid(df.nsmallest(12, "luminance_std"), ncols=6,
          title_fn=lambda r: f'{r["label"]}\nstd={r["luminance_std"]:.1f}',
          suptitle="Lowest contrast images")

In [ ]:
# Lowest contrast images: image and grayscale histogram side by side
low_contrast = df.nsmallest(12, "luminance_std")

for start in range(0, len(low_contrast), 6):
    chunk = low_contrast.iloc[start:start + 6]
    fig, axes = plt.subplots(2, 6, figsize=(18, 5))
    for i, (_, row) in enumerate(chunk.iterrows()):
        img = imread_unicode(row["file_path"])
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        show_image(img, title=f'{row["label"]}\nstd={row["luminance_std"]:.1f}', ax=axes[0, i])
        axes[1, i].hist(gray.ravel(), bins=64, range=(0, 256), color="gray")
        axes[1, i].set_xlabel("Intensity")
    axes[1, 0].set_ylabel("Pixel count")
    plt.tight_layout()
    plt.show()

The lowest contrast images have a single narrow peak in the histogram, which comes from a uniform background (grass, wall, asphalt, or a smooth surface). The insect occupies few pixels and has a gray tone similar to the background, so it does not form a second peak. Even so, the insect is still easy to see in most images, since it differs from the background in color. One exception is a very dark beetle image, where the peak is close to zero because the whole image is dark.

# 4. Dataset splitting


In [ ]:
# Cleaning duplicates

df_clean = df.drop_duplicates(subset="hash", keep="first").reset_index(drop=True)

In [ ]:
# Splitting the dataset to 70% training/15% validation/15% testing
SPLITTING_SEED = 14

train_idx, rest_idx = train_test_split(
    df_clean.index, test_size=0.3,
    stratify=df_clean["label"], random_state=SPLITTING_SEED
)

val_idx, test_idx = train_test_split(
    rest_idx, test_size=0.5,
    stratify=df_clean.loc[rest_idx, "label"], random_state=SPLITTING_SEED
)

df_clean["split"] = "train"
df_clean.loc[val_idx,"split"] = "val"
df_clean.loc[test_idx,"split"] = "test"

# Pictures split per class
pd.crosstab(df_clean["label"], df_clean["split"], margins=True)

# Save split datset for reproducibility
df_clean.to_csv('df_clean_crop_pests.csv', index=False)

Why do we split the dataset like this?

The dataset has no pre-defined split, so we created one. After removing the 6 exact duplicates, the 5,488 images were divided into 70% training (3,841), 15% validation (823) and 15% test (824), stratified by class with a fixed seed (14). The training set is used to fit the models, the validation set to tune hyperparameters and for early stopping, and the test set is used only once at the end. Stratification keeps the class proportions in all subsets, so that small classes are not under-represented in the validation or test sets by chance. Duplicates were removed before splitting so that the same image cannot be in both training and test.

Goodfellow et al. (2016) state that typically about 80% of the training data is used for training and 20% for validation, so we would have preferred to use about 80% for training. However, some classes are small: earthworms has only 323 images, and with 10% it would have about 32 images in each of the validation and test sets. Raising both to 15% gives 48 to 49 images, which makes the evaluation of the small classes more reliable. The cost is about 550 fewer training images.

Joseph (2022) shows that the optimal split ratio depends on the complexity of the model, at least for linear regression, so there is no single ideal ratio. Whether 70/15/15 was the best choice can only be verified empirically with another split, such as 80/10/10, which we did not do here.

# 5. Image Preprocessing


In [ ]:
# Padding color equals to mean of all classes (BGR color) of the training images
train_paths = df_clean.loc[df_clean["split"] == "train", "file_path"]
train_means = np.array([imread_unicode(p).reshape(-1, 3).mean(axis=0) for p in train_paths])

PAD_COLOR = tuple(int(round(v)) for v in train_means.mean(axis=0))

print("Padding color (B, G, R):", PAD_COLOR)

I chose the mean color of the training images, computed over all classes together, as the padding color. My goal in this step is only to adjust the size of the images, without distorting them, using nothing but padding. I did not choose to extend the border colors, because that would insert many repeated colors into the images and could confuse the models. With a single color for every image, the models can learn that this background is a standard, general filler, the same in all cases, added only so that all images follow the 224x224 pattern. Almost all the images have their longest side close to 300 pixels, so resizing to 224x224 does not require much change. The shape and texture of the insects seem to be more relevant than color, which is repetitive because of the background. The 224x224 size is also the standard input of the MobileNetV2 (Sandler et al., 2018), which keeps enough resolution to train the models without increasing the computational cost too much.

In [ ]:
# Letterbox resize
IMG_SIZE = 224 # 224 x 224

# Keeps the aspect ratio and fills the rest with PAD_COLOR
def letterbox(img_bgr, size=IMG_SIZE, pad_color=PAD_COLOR):
    h, w = img_bgr.shape[:2]
    scale = min(size / h, size / w, 1.0)  # Never upscale the pictures
    new_w, new_h = max(1, round(w * scale)), max(1, round(h * scale))
    resized = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.full((size, size, 3), pad_color, dtype=np.uint8) # Create square canvas with the padding color
    top, left = (size - new_h) // 2, (size - new_w) // 2 
    canvas[top:top + new_h, left:left + new_w] = resized # Fits the resized image into the center of the canvas
    return canvas  # <- missing

In [ ]:
# Testing the letterbox
test_rows = pd.concat([
    df_clean.nsmallest(2, "aspect_ratio"), # 2 tallest
    df_clean.nlargest(2, "aspect_ratio"), # 2 widest
    df_clean[df_clean[["height", "width"]].max(axis=1) == 512].head(2), # 512 px cases
])

fig, axes = plt.subplots(2, len(test_rows), figsize=(2.6 * len(test_rows), 5.5))

for j, (_, row) in enumerate(test_rows.iterrows()):
    img = imread_unicode(row["file_path"])
    show_image(img, f'{row["label"]} {row["width"]}x{row["height"]}', ax=axes[0, j])
    show_image(letterbox(img), "224x224", ax=axes[1, j])
plt.tight_layout()
plt.show()

Letterbox worked as intended: images reach 224x224 without stretching, and the insect keeps its shape.

Some images already have a black frame in the original photo (the caterpillar above). This is not the norm in the dataset, only an occasional case. We kept them unchanged, since the frame may only add some noise to HOG and LBP.

In [ ]:
# Store in RGB, the order expected by matplotlib and the CNNs
images = np.empty((len(df_clean), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) # 2**8 memory space
for i, p in enumerate(df_clean["file_path"]):
    images[i] = cv2.cvtColor(letterbox(imread_unicode(p)), cv2.COLOR_BGR2RGB) # BGR -> RGB

np.save("images_224_rgb.npy", images) #Save images to disk as cache (working directory)
print(images.shape, images.dtype, f"{images.nbytes / 1e6:.0f} MB")

# 6. Traditional approach


I chose HOG (Histogram of Oriented Gradients) and LBP (Local Binary Pattern) because the exploratory analysis showed that color is a weak feature in this dataset. The background dominates the color histograms, and classes with very similar mean color have very different shapes. Shape and texture are what separate the classes. HOG describes the shape, because it counts the direction of the edges in each region of the image, and LBP describes the texture of the surface, because it compares each pixel with its neighbors.

In [ ]:
# Feature extractors
# HOG for shape and edges
# LBP for texture

# HOG
# 32x32 cells -> 7x7 cells and 2x2 blocks -> 7 - 2 + 1 = 6 positions possible -> 6*6*2*2*9 = 1296 features
HOG_PARAMS = dict(orientations=9, pixels_per_cell=(32, 32), cells_per_block=(2, 2), block_norm="L2-Hys")

# LBP 8 neighbours, radii 1, 2 and 3 (10 codes each), histogram per cell of a 4x4 grid -> 3*160 = 480 features
LBP_P, LBP_RADII, LBP_GRID = 8, (1, 2, 3), 4 # 8 neighbours, 10 groups, 4x4 grid
LBP_BINS = LBP_P + 2

def hog_features(gray):
    return hog(gray, **HOG_PARAMS) # ** unpacks the dict into keyword arguments

def lbp_features(gray):
    step = gray.shape[0] // LBP_GRID
    feats = []
    for radius in LBP_RADII:
        lbp = local_binary_pattern(gray, LBP_P, radius, method="uniform")
        for r in range(LBP_GRID):
            for c in range(LBP_GRID):
                cell = lbp[r * step:(r + 1) * step, c * step:(c + 1) * step]
                hist, _ = np.histogram(cell, bins=LBP_BINS, range=(0, LBP_BINS))
                feats.append(hist / hist.sum())
    return np.concatenate(feats)

def extract(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    return hog_features(gray), lbp_features(gray)


In [ ]:
# Extracting HOG and LBP for all images in parallel processing
results = Parallel(n_jobs=6)(delayed(extract)(img) for img in images)
X_hog_lbp = np.array([np.concatenate(r) for r in results], dtype=np.float32)
print("HOG+LBP:", X_hog_lbp.shape)  # expected (5488, 1776)

I used the two features concatenated, in a single vector per image, because each one looks at something different, shape in HOG and texture in LBP, so together they give the classifiers more information than either one alone. Concatenation is also the simplest way to combine them, since the classifiers receive one vector per image. With the current parameters (HOG with 16x16 pixel cells, LBP with a 4x4 grid), HOG has 6,084 features and LBP has 160, so HOG may have more influence than LBP on the classifiers. We did not measure this, and these parameters can still be changed if the results of the training are not satisfactory.

In [ ]:
# Labels and split masks

y = df_clean["label"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
is_train = (df_clean["split"] == "train").to_numpy()
is_val = (df_clean["split"] == "val").to_numpy()
is_test = (df_clean["split"] == "test").to_numpy()

# Scaler fitted only on the training set
scaler = StandardScaler().fit(X_hog_lbp[is_train])
X_train = scaler.transform(X_hog_lbp[is_train])
X_val = scaler.transform(X_hog_lbp[is_val])
X_test = scaler.transform(X_hog_lbp[is_test])

y_train, y_val, y_test = y[is_train], y[is_val], y[is_test]
print("train:", X_train.shape, "| val:", X_val.shape, "| test:", X_test.shape)


For the traditional approach we will train two classifiers. The SVM (Support Vector Machine) looks for the boundary that separates the classes with the largest margin between them, measuring how close the feature vectors are to each other. The Random Forest is a set of decision trees, each one trained with a random part of the data, that vote for the final class.

I used StandardScaler because the SVM is based on the distance between the feature vectors, and features with larger values would count more in this distance only because of their scale, not because they are more useful. The scaler subtracts the mean and divides by the standard deviation of each feature, so all the 6,244 features of HOG and LBP start with the same scale. The Random Forest does not need this step, but it is not harmed by it, so both classifiers use the same scaled features. The scaler was fitted only on the training set and then applied to the validation and test sets. If it were fitted with all the data, information from the validation and test sets would leak into the preprocessing and the evaluation would be too optimistic. One possible problem, which we did not measure, is that the scaler can amplify the noise of features with very small variation.

In [ ]:
# Saving features (scaled) and labels so the classifiers can be trained without re-extracting
np.savez_compressed(
    "features_traditional.npz",
    X_train=X_train, X_val=X_val, X_test=X_test,
    y_train=y_train, y_val=y_val, y_test=y_test,
)

In [ ]:
# Training the two traditional classifiers

svm = SVC()
svm.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=300, n_jobs=6, random_state=14)
rf.fit(X_train, y_train)

In [ ]:
# Accuracy and macro F1 on the validation set

for name, model in [("SVM", svm), ("Random Forest", rf)]:
    pred = model.predict(X_val)
    acc = accuracy_score(y_val, pred)
    f1 = f1_score(y_val, pred, average="macro")
    print(name, "accuracy", round(acc, 3), "macro F1", round(f1, 3))

In [ ]:
# Confusion matrix of the best model on the validation set

best = svm  # change to rf if it has the higher macro F1
ConfusionMatrixDisplay.from_estimator(
    best, X_val, y_val, display_labels=CLASSES, xticks_rotation=45, cmap="Blues"
)
plt.show()

In [ ]:
# Results table with the metrics of every model on train, validation and test

RESULTS_CSV = "results_all_models.csv"
RESULT_COLS = ["model", "approach", "split", "accuracy", "precision_macro", "recall_macro", "f1_macro", "config"]

# Loads the results of previous runs, a model is replaced when it is trained again
results_df = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame(columns=RESULT_COLS)

def show_results():
    # Explains the metrics and shows all models, best validation macro F1 first
    print("Metrics from 0 to 1, higher is better.")
    print("Accuracy is the share of correct predictions.")
    print("Precision, recall and F1 are macro averages, the mean of the 12 classes with the same weight for each class.")
    print("The models are ranked by the validation F1 macro, the test set is only reported.")
    metrics = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
    splits = ["train", "val", "test"]
    table = results_df.pivot(index="model", columns="split", values=metrics)
    table = table.reindex(columns=pd.MultiIndex.from_product([metrics, splits], names=["metric", "split"]))
    table = table.sort_values(("f1_macro", "val"), ascending=False)
    display(table.round(3))

def log_result(name, approach, preds, config="", show=True):
    # preds has one (y_true, y_pred) pair for each split (train, val and test)
    # Computes the metrics, saves them in the CSV and shows the table unless show is False
    global results_df
    rows = []
    for split, (y_true, y_pred) in preds.items():
        rows.append({
            "model": name, "approach": approach, "split": split, "config": config,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_true, y_pred, average="macro"),
        })
    old = results_df[results_df["model"] != name]
    results_df = pd.DataFrame(rows) if old.empty else pd.concat([old, pd.DataFrame(rows)], ignore_index=True)
    results_df.to_csv(RESULTS_CSV, index=False)
    if show:
        show_results()


In [ ]:
# Metrics of the traditional models on train, validation and test
# Test is only reported, the models are chosen with the validation set
for name, model in [("SVM (HOG+LBP)", svm), ("Random Forest (HOG+LBP)", rf)]:
    log_result(name, "traditional", {
        "train": (y_train, model.predict(X_train)),
        "val": (y_val, model.predict(X_val)),
        "test": (y_test, model.predict(X_test)),
    }, show=False)

# One table with both models
show_results()

# Saves the traditional models in the same folder of the deep learning models
# The scaler is needed to use the SVM on new images
os.makedirs("models", exist_ok=True)
joblib.dump(svm, "models/svm_hog_lbp.joblib")
joblib.dump(rf, "models/random_forest_hog_lbp.joblib", compress=3)  # compressed to use less disk space
joblib.dump(scaler, "models/scaler_hog_lbp.joblib")

# 7. Data augmentation


In [ ]:
# Data augmentation pipeline, online and only for the training set

# Padding color of the letterbox in RGB (PAD_COLOR is BGR)
PAD_RGB = PAD_COLOR[::-1]

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    # Rotation fills the corners with the padding color
    A.SafeRotate(limit=30, p=0.5, fill=PAD_RGB),
    # Crop keeps at least 70% of the image
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0), ratio=(0.9, 1.1), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    # Only one degradation (blur, noise or JPEG) in 20% of the images
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5)),
        A.GaussNoise(std_range=(0.02, 0.08)),
        A.ImageCompression(quality_range=(50, 95)),
    ], p=0.2),
], seed=SPLITTING_SEED)

# Same image augmented several times (images are already RGB)
sample = images[np.flatnonzero(is_train)[0]]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes[0, 0].imshow(sample)
axes[0, 0].set_title("Original", fontsize=9)
for ax in axes.flat[1:]:
    ax.imshow(train_transform(image=sample)["image"])
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Class weights to compensate the imbalance (rarer classes weigh more in the loss)

weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = {i: w for i, w in enumerate(weights)}

for i, c in enumerate(CLASSES):
    print(f"{c:12s} {class_weight[i]:.2f}")

# 8. Deep learning approach


In [ ]:
# Deep learning setup with Keras 3 and the PyTorch backend
# The backend was set in the first cell, before importing keras

assert keras.backend.backend() == "torch", "Restart the kernel: keras was imported before the backend was set"
keras.utils.set_random_seed(SPLITTING_SEED)
print("Keras", keras.__version__, "| CUDA:", torch.cuda.is_available())

keras.mixed_precision.set_global_policy("mixed_float16")  # half precision uses less memory and is faster

# Images are uint8 RGB 224x224, each model rescales them in its first layer
X_train_img, y_train_dl = images[is_train], y[is_train]
X_val_img, y_val_dl = images[is_val], y[is_val]
X_test_img, y_test_dl = images[is_test], y[is_test]
NUM_CLASSES = len(CLASSES)

# Use 0 to be exactly reproducible, but it is slower
N_WORKERS = 2  # CPU threads that augment the batches, fewer threads is less load on the PC

# Online augmentation, each batch is augmented when the model asks for it
class AugmentedSequence(keras.utils.PyDataset):
    def __init__(self, x, y, batch_size, transform=None, **kwargs):
        super().__init__(**kwargs)
        self.x, self.y, self.batch_size, self.transform = x, y, batch_size, transform
        self.indices = np.arange(len(x))

    def __len__(self):
        return len(self.x) // self.batch_size  # drops the last incomplete batch, it had only 1 image

    def __getitem__(self, idx):
        batch = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        if self.transform is None:
            xb = self.x[batch].astype("float32")
        else:
            xb = np.stack([self.transform(image=self.x[i])["image"] for i in batch]).astype("float32")
        yb = self.y[batch]
        # Class weights are passed as sample weights
        sw = np.array([class_weight[c] for c in yb], dtype="float32")
        return xb, yb, sw

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

In [ ]:
# Training function that fits a model, plots the learning curves and saves the metrics
# Validation predictions of each model, used in the error analysis
val_predictions = {}

# Frees memory at the end of each epoch, without it the GPU memory grows epoch after epoch
class CleanMemory(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        gc.collect()  # frees Python objects that were not released yet
        torch.cuda.empty_cache()  # gives the unused GPU memory back
        print(f"GPU memory in use {torch.cuda.memory_allocated() / 1e9:.1f} GB")

def predict_labels(model, x):
    return model.predict(x, batch_size=64, verbose=0).argmax(axis=1)  # small batches keep the memory peak low

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for ax, metric in zip(axes, ["loss", "accuracy"]):
        ax.plot(history.history[metric], label="train")
        ax.plot(history.history["val_" + metric], label="validation")
        ax.set_xlabel("epoch")
        ax.set_title(f"{title}: {metric}", fontsize=9)
        ax.legend()
    plt.tight_layout()
    plt.show()

def train_model(name, model, optimizer_name, lr, batch_size, epochs, augment=True):
    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=lr)
    else:
        optimizer = keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    train_seq = AugmentedSequence(X_train_img, y_train_dl, batch_size,
                                  transform=train_transform if augment else None,
                                  workers=N_WORKERS, use_multiprocessing=False)
    # Learning rate scheduler and early stopping on the validation loss
    callbacks = [
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),  # halves the learning rate when the loss stops improving
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),  # stops training and keeps the best weights
        CleanMemory(),  # frees memory at every epoch
    ]

    start = time.time()
    history = model.fit(train_seq, validation_data=(X_val_img, y_val_dl),
                        epochs=epochs, callbacks=callbacks, verbose=2)
    seconds = time.time() - start
    plot_history(history, name)

    # Metrics on the original images, without augmentation
    val_pred = predict_labels(model, X_val_img)
    val_predictions[name] = val_pred
    config = f"{optimizer_name} lr={lr} bs={batch_size} aug={augment} epochs={len(history.history['loss'])} {seconds:.0f}s"
    log_result(name, "deep learning", {
        "train": (y_train_dl, predict_labels(model, X_train_img)),
        "val": (y_val_dl, val_pred),
        "test": (y_test_dl, predict_labels(model, X_test_img)),
    }, config=config)

    # Saves the model with the best validation weights
    os.makedirs("models", exist_ok=True)
    model.save("models/" + re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_").lower() + ".keras")
    gc.collect()  # frees memory before the next model
    torch.cuda.empty_cache()
    return history

In [ ]:
# Model 1 is a small CNN trained from scratch
def build_cnn_scratch():
    keras.utils.set_random_seed(SPLITTING_SEED)
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    # Pixels to [0, 1]
    x = keras.layers.Rescaling(1 / 255)(inputs)
    x = keras.layers.Conv2D(32, 3, strides=2, padding="same", use_bias=False)(x)  # stride 2 halves the image, so the next layers are lighter
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.ReLU()(x)
    x = keras.layers.MaxPooling2D()(x)
    # 3 blocks of 2 convolutions
    for filters in (64, 128, 256):
        for _ in range(2):
            x = keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
            x = keras.layers.BatchNormalization()(x)
            x = keras.layers.ReLU()(x)
        x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.4)(x)
    outputs = keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)  # output in float32, needed by the mixed precision
    return keras.Model(inputs, outputs, name="cnn_scratch")

# Model 2 is MobileNetV2 with ImageNet weights (transfer learning)
# The feature extractor is frozen at first
def build_mobilenet():
    keras.utils.set_random_seed(SPLITTING_SEED)
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = False
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    # Pixels to [-1, 1], same as the MobileNetV2 preprocess_input
    x = keras.layers.Rescaling(1 / 127.5, offset=-1)(inputs)
    # Batch norm layers stay in inference mode
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)  # output in float32, needed by the mixed precision
    return keras.Model(inputs, outputs, name="mobilenetv2"), base

# Fine-tuning unfreezes the last layers of the base, batch norm layers stay frozen
def unfreeze_top_layers(base, n_layers):
    base.trainable = True
    for layer in base.layers[:-n_layers]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

In [ ]:
# Training model 1 (CNN from scratch) with Adam, lr 1e-3, batch 64 and augmentation
EPOCHS_SCRATCH = 40  # early stopping usually stops before

model_cnn = build_cnn_scratch()
model_cnn.summary()
train_model("CNN from scratch", model_cnn, "adam", 1e-3, 64, epochs=EPOCHS_SCRATCH)

In [ ]:
# Training model 2 (MobileNetV2) with the feature extractor frozen, only the classifier is trained
# Hyperparameter experiment with optimizer, learning rate and batch size
# The configurations are compared on the validation set
EPOCHS_HEAD = 30
HP_GRID = [("adam", 1e-3, 64), ("adam", 1e-4, 64), ("sgd", 1e-2, 64), ("adam", 1e-3, 32)]

hp_models = {}
for opt_name, lr, bs in HP_GRID:
    name = f"MobileNetV2 frozen ({opt_name}, lr={lr}, bs={bs})"
    model, base = build_mobilenet()
    train_model(name, model, opt_name, lr, bs, epochs=EPOCHS_HEAD)
    hp_models[name] = (model, base, opt_name, lr, bs)

# Best configuration by validation macro F1
val_rows = results_df[(results_df["split"] == "val") & (results_df["model"].isin(hp_models))]
best_hp_name = val_rows.sort_values("f1_macro").iloc[-1]["model"]
print("Best configuration:", best_hp_name)

In [ ]:
# Fine-tuning of the best configuration, now also updating the top layers of MobileNetV2
# The learning rate is 10x smaller
EPOCHS_FT = 30

model_ft, base_ft, opt_name, lr, bs = hp_models[best_hp_name]
unfreeze_top_layers(base_ft, 30)
train_model("MobileNetV2 fine-tuned", model_ft, opt_name, lr / 10, bs, epochs=EPOCHS_FT)

In [ ]:
# Qualitative analysis of the best deep learning model on the validation set

dl_val = results_df[(results_df["approach"] == "deep learning") & (results_df["split"] == "val")]
best_dl = dl_val.sort_values("f1_macro").iloc[-1]["model"]
pred = val_predictions[best_dl]
print("Best deep learning model:", best_dl)
print(f"Validation, {int((pred == y_val_dl).sum())} correct and {int((pred != y_val_dl).sum())} wrong out of {len(y_val_dl)} images")
print(classification_report(y_val_dl, pred, target_names=CLASSES, digits=3))

disp = ConfusionMatrixDisplay.from_predictions(
    y_val_dl, pred, display_labels=CLASSES, xticks_rotation=45, cmap="Blues"
)
disp.ax_.set_title(f"Validation - {best_dl}", fontsize=10)
plt.show()

# Random sample of images, red titles for errors and green titles for correct predictions
# The title also shows the model and how many images it got right and wrong
def show_samples(x, y_true, y_pred, indices, color, title):
    n_wrong = int((y_true != y_pred).sum())
    n_total = len(y_true)
    fig, axes = plt.subplots(3, 4, figsize=(10, 10))
    fig.suptitle(f"{title}\n{best_dl} | {n_total - n_wrong} correct and {n_wrong} wrong out of {n_total} images",
                 fontsize=12, color=color)
    for ax, i in zip(axes.flat, indices):
        ax.imshow(x[i])
        ax.set_title(f"True: {CLASSES[y_true[i]]}\nPred: {CLASSES[y_pred[i]]}", fontsize=9, color=color)
    for ax in axes.flat:
        ax.axis("off")
    fig.subplots_adjust(top=0.87, bottom=0.02, left=0.02, right=0.98, hspace=0.55, wspace=0.08)
    plt.show()

rng = np.random.default_rng(SPLITTING_SEED)
wrong = np.flatnonzero(pred != y_val_dl)
right = np.flatnonzero(pred == y_val_dl)
show_samples(X_val_img, y_val_dl, pred, rng.choice(wrong, size=min(12, len(wrong)), replace=False),
             "red", "Validation - Misclassified Images")
show_samples(X_val_img, y_val_dl, pred, rng.choice(right, size=12, replace=False),
             "green", "Validation - Correctly Classified Images")

In [ ]:
# Qualitative analysis of the best deep learning model on the test set
# Test is only for the final report, nothing is changed after looking at it
# The predictions come from the saved file of the best model chosen on validation
best_file = "models/" + re.sub(r"[^A-Za-z0-9]+", "_", best_dl).strip("_").lower() + ".keras"
best_model = keras.models.load_model(best_file)
test_pred = predict_labels(best_model, X_test_img)
print("Best deep learning model:", best_dl)
print(f"Test, {int((test_pred == y_test_dl).sum())} correct and {int((test_pred != y_test_dl).sum())} wrong out of {len(y_test_dl)} images")
print(classification_report(y_test_dl, test_pred, target_names=CLASSES, digits=3))

disp = ConfusionMatrixDisplay.from_predictions(
    y_test_dl, test_pred, display_labels=CLASSES, xticks_rotation=45, cmap="Blues"
)
disp.ax_.set_title(f"Test - {best_dl}", fontsize=10)
plt.show()

rng = np.random.default_rng(SPLITTING_SEED)
wrong = np.flatnonzero(test_pred != y_test_dl)
right = np.flatnonzero(test_pred == y_test_dl)
show_samples(X_test_img, y_test_dl, test_pred, rng.choice(wrong, size=min(12, len(wrong)), replace=False),
             "red", "Test - Misclassified Images")
show_samples(X_test_img, y_test_dl, test_pred, rng.choice(right, size=12, replace=False),
             "green", "Test - Correctly Classified Images")

In [ ]:
# Final comparison of the traditional and deep learning models
# All metrics are also saved in results_all_models.csv
# The best model is chosen by the validation macro F1, the test set is only reported
show_results()

# Digest


```
Add your text here.
```

# References

- Goodfellow, I., Bengio, Y. and Courville, A. (2016). Deep Learning. MIT Press.
- Joseph, V. R. (2022). Optimal ratio for data splitting. Statistical Analysis and Data Mining, 15, 531–538.
- Sandler, M., Howard, A., Zhu, M., Zhmoginov, A. and Chen, L.-C. (2018). MobileNetV2: Inverted Residuals and Linear Bottlenecks. In IEEE Conference on Computer Vision and Pattern Recognition (CVPR), pp. 4510–4520. https://arxiv.org/abs/1801.04381
